In [ ]:
from datasets import load_dataset, get_dataset_split_names
from pprint import pprint
from collections import Counter

In [ ]:
#load each of the configs into local cache
configs = ["sft_sequence_classification",
"sft_dna_computation",
"rl_mix",
]

datasets = {name : load_dataset("HuggingFaceBio/FineDNA", name, split="train", streaming=False) for name in configs}

Generating train split: 16396 examples [00:00, 124582.08 examples/s]
Generating train split: 44925 examples [00:00, 508333.62 examples/s]
Generating train split: 98433 examples [00:00, 205210.32 examples/s]


In [22]:
computation_ds = datasets["sft_dna_computation"]
classification_ds = datasets["sft_sequence_classification"]

wanted_domains = ["dna_computation", "sequence_classification", 
                  "biology_qa", "instruction_following"]

rl_ds = datasets["rl_mix"].filter(lambda row: row['domain'] in wanted_domains)

Filter: 100%|██████████| 98433/98433 [00:00<00:00, 113361.96 examples/s]


In [116]:
#check available splits in each config
for config in configs:
    splits = get_dataset_split_names('HuggingFaceBio/FineDNA', config)
    print(f"{config} : {splits}")

sft_sequence_classification : ['train']
sft_dna_computation : ['train']
rl_mix : ['train']


In [24]:
#inspect the classification dataset columns and rows
for row in classification_ds.select(range(2)):
    pprint(row)
    pprint("____________________")

{'chat_template_kwargs': {'tokenizer_kwargs': {'auto_dna_tags': True}},
 'domain': 'sequence_classification',
 'messages': [{'content': 'Select the sequence corresponding to the YY1 '
                          'consensus.\n'
                          '\n'
                          'Options:\n'
                          'A. CCCAATGTAAACATGAGTCT\n'
                          'B. GTGGTTTGGGATTCCTACGA\n'
                          'C. CCGGTACAAGATGGCGGCCG\n'
                          'D. TGCGTCACTTCCGGTAACCG',
               'role': 'user'},
              {'content': 'YY1 sites often fit the consensus GCCAT, then any '
                          'base, then TT, where “N” can be any DNA base, and '
                          'the motif can occur on either strand. **C** fits '
                          'best: its reverse complement contains GCCATCTT, an '
                          'exact match to that consensus. A has a shorter ACAT '
                          'fragment, while B and D lack a com

In [25]:
#inspect the DNA computation dataset columns and rows
for row in computation_ds.select(range(2)):
    pprint(row)
    pprint("____________________")

{'chat_template_kwargs': {'tokenizer_kwargs': {'auto_dna_tags': True}},
 'domain': 'dna_computation',
 'messages': [{'content': 'Motif dictionary: GATA family=WGATAR; ETS '
                          'family=GGAA; bHLH family=CANNTG; POU '
                          'family=ATGCAAAT; NF-kappaB family=GGGRNNYYCC; AP-1 '
                          'family=TGASTCA\n'
                          "Original: 5'-GGGTGAGTCAATTCACTATT-3'\n"
                          "Modified: 5'-GGGTAAGTCAATTCACTATT-3'\n"
                          'Answer one of: Retained | Lost | Gained | Switched\n'
                          'Using the motif dictionary, classify the AP-1 '
                          'family binding site across the two sequences.',
               'role': 'user'},
              {'content': 'The AP-1 motif is TGASTCA, where S can be G or C. '
                          'Thus, the matching sequences are TGAGTCA and '
                          'TGACTCA.\n'
                          '\n'
                

In [26]:
#inspect the RL dataset columns and rows
for row in rl_ds.select(range(2)):
    pprint(row)
    pprint("___________________")

{'answer_text': None,
 'domain': 'instruction_following',
 'gold_answer_json': None,
 'prompt': [{'content': 'Please follow these instructions in your response:\n'
                        '- In your response, the letter b should appear at '
                        'least 3 times.\n'
                        '- There should be 4 paragraphs. Paragraphs and only '
                        'paragraphs are separated with each other by two new '
                        "lines as if it was '\\n\\n' in python. Paragraph 2 "
                        'must start with word doubt.\n'
                        '\n'
                        '帮我写一份锐捷AP设备产品介绍以及对比TP设备，企业为什么更适合选择锐捷设备的优势的分析宣传',
             'role': 'user'}],
 'task_group': 'if',
 'task_type': None,
 'verifier_metadata': {'instruction_id_list': ['keywords:letter_frequency',
                                               'length_constraints:nth_paragraph_first_word'],
                       'kwargs': [{'let_frequency': 3,
                       

In [ ]:
# Store all the types of task-groups and sub-tasks in our RL data into './rl-data-groups'
from typing import Any, cast
from pathlib import Path

def find_groups_and_types(domain):
    task_groups = Counter()
    task_types = Counter()
    rl_domain = rl_ds.filter(lambda row: row['domain']==domain)

    for item in rl_domain:
        row = cast(dict[str, Any], item)
        task_groups[row.get('task_group')] += 1
        task_types[(row.get('task_group'), row.get('task_type'))] += 1

    task_types = sorted(task_types.items(), key=lambda item: (item[0][0] or "", item[1]))

    with open(Path(f"./rl-data-groups/rl-groups-{domain}.txt"), 'w') as f:
        f.write(f"Groups:\n")
        for group, count in task_groups.most_common():
            f.write(f"{str(group)}: {count}\n")
        f.write("__________________________\n")
        f.write("Types:\n")
        for (group, task_type), count in task_types:
            f.write(f"{group}/{task_type}: {count}\n")
    return task_groups, task_types
task_group_types_counters = []

for domain in wanted_domains:
    _, type_counter = find_groups_and_types(domain)
    task_group_types_counters.append(type_counter)

In [ ]:
# function to check whether we can verify an answer directly using the "answer_text" field
def check_task_group_verifiable(filtered_rl_ds, domain, task_group, task_type):
        answer_text_is_reward = True

        for row in filtered_rl_ds:
            row = cast(dict[str, Any], row)
            has_answer = row.get("answer_text") not in (None, "")
            has_metadata = row.get("verifier_metadata") not in (None, "")
            has_gold_answer = row.get("gold_answer_json") not in (None, "")

            if not has_answer or has_metadata or has_gold_answer:
                answer_text_is_reward = False
        return answer_text_is_reward

In [ ]:
# checking the verifiability of dna_computation tasks
domain = wanted_domains[0]
verifiable_task_groups = ["basic_dna", "bio_mcq", "genomeqa", "seqrecqa_position"]

for task_types_counter in task_group_types_counters:
    for (task_group, task_type), _ in task_types_counter:
        if task_group in verifiable_task_groups:
                    continue
                    
        filtered_rl_ds = rl_ds.filter(lambda row: row["domain"]==domain and row["task_group"]==task_group and row["task_type"]==task_type)
        
        answer_text_is_reward = check_task_group_verifiable(filtered_rl_ds, domain, task_group, task_type)
        print(f"{domain}/{task_group}/{task_type}: {answer_text_is_reward}")
        print(len(filtered_rl_ds))
        for row in filtered_rl_ds.select(range(4)):
            print(row)
        print("_______________________\n\n")

dna_computation/dna_skills/None: False
6520
{'prompt': [{'content': 'Reference CDS:\nATGGGGAAGATCACCTTCTATGAGGACCGCGGCTTCCAGGGCCGCCACTATGAGTGCAGCACCGACCACTCCAACCTGCAGCCCTACTTCAGCCGCTGCAACTCTGTGCGCGTGGACAGTGGCTGCTGGATGCTCTATGAGCAGCCCAACTTCACGGGCTGCCAGTACTTCCTGCGTCGCGGGGACTACCCTGACTACCAGCAGTGGATGGGTTTCAGTGACTCTGTCCGCTCCTGCCGCCTCATCCCCCACGCCGGCTCTCACAGGATCAGACTGTACGAGAGGGAAGAGTACAGAGGCCAGATGATAGAGTTCACGGAGGACTGCCCCTCTCTCCAGGACCGATTCCACTTCAATGAGATCTACTCCCTCAATGTGCTAGAGGGCTGCTGGGTCCTCTACGACATGACCAACTACCGGGGAAGGCAGTACTTGCTCAGACCCGGGGAGTACAGGCGCTACCACGACTGGGGCGCCATGAATGCCAGGGTGGGCTCTCTGAGGAGAGTCATGGATTTCTACTGA\n\nClone:\nATGGGGAAGATCACCTTCTATGAGGACCGCGGCTTCCAGGGCCGCCACTATGAGTGCAGCACCGACCACTCCAACCTGCAGCCCTACTTCAGCCGCTGCAACTCTGTGCGCGTGGACAGTGGCTGCTGGATGCTCTATGAGCAGCCCAACTTCACGGGCTGCCAGTACTTCCTGCGTCGCGGGGACTACCCTGACTACCAGCAGTGGATGGGTTTCAGTGACTCTGTCCGCTCCTGCCGCCTCATCCCCCACGCCGGCTCTCACAGGATCAGACTGTACGAGAGGGAAGAGTACAGAGGCCAGATGATAGAGTTCACGGAGGACTGCCCCTCTCTCCAGGACCGATTCCACTTCAATGAGATCTACTCCCTCAATGTG

In [ ]:
# checking the verifiability of answers inside the sequence classification domain
domain = wanted_domains[1]
verifiable_task_groups = ["genomeqa"]
for task_types_counter in task_group_types_counters:
    for (task_group, task_type), _ in task_types_counter:
        if task_group not in verifiable_task_groups:
                    continue
                    
        filtered_rl_ds = rl_ds.filter(lambda row: row["domain"]==domain and row["task_group"]==task_group and row["task_type"]==task_type)

        answer_text_is_reward = check_task_group_verifiable(filtered_rl_ds, domain, task_group, task_type)
        print(f"{domain}/{task_group}/{task_type}: {answer_text_is_reward}")
        print(len(filtered_rl_ds))
        
        for row in filtered_rl_ds.select(range(4)):
            print(row)
        print("_______________________\n\n")

sequence_classification/genomeqa/None: True
14032
{'prompt': [{'content': 'You are an expert in epigenomics. You relate human DNA sequences to histone modifications and chromatin state in K562 cells.', 'role': 'system'}, {'content': "Does this Human DNA sequence typically reside in open or closed chromatin in K562 cells?\nSequence: AGGAAAAGGTAGATCAATTAATGAAGATTAGTGCTTCCAACTGGTTGTTGTAGTAGAATGTGAGGGTGGTAACTTTTGAGAGACCAGCCAGTAGTCATAGTGGCAAGCAAAGATGATCTTAGCTGAGGTGAAGAGAAGGAAAGAAAAGACAGAGAAAATAGGGAACTTGTAACAAGGAGGAGGCTTAGATAAACACTATAATACTTACCATAGTTGATTAGCAATGTACCAATGAGGAATTTAATATAGTCAGTAGAAATGAAGGGAGATTAGGAAATGTGAATGCATACAGAATAGTAACGGTCTCCCTGTAAATGTGAAAAAGCAATTTTCTGTGATAACTATAAACATGATAGTGATGATGATGTTGACAATTAAAATAACGGTGACTGTTAAGAAGATAGAAGGGCAAAATTTGACCACCAGAATAAAATCAAGAGCCAAATATGGGCCACTGAGGTAAACAGAATGGTTATTCTAGAATTCTTATTAAATATTAGCATATTTCTCTGTTTTCCTTGCTGCCCTGCATTGGCAAGGGAAAAAGAAGTGCATAAGTATAATTTTTCTTATTGGAAAGAATTTTTCCTGTATACTTTCAAGTCAGTAAAATATTGCATGTCTATTCCATATTCCTGATGGGCTTTGAAAGATGTCATAATGGGC

In [ ]:
# checking the verifiability of answers inside the biology_qa domain
domain_index = 2
domain = wanted_domains[domain_index]
verifiable_task_groups = []
task_types_counter = task_group_types_counters[domain_index]

for (task_group, task_type), _ in task_types_counter:
    print(domain, task_group, task_type)
    if task_group in verifiable_task_groups:
                continue
                
    filtered_rl_ds = rl_ds.filter(lambda row: row["domain"]==domain and row["task_group"]==task_group and row["task_type"]==task_type)
    print(len(filtered_rl_ds))
    answer_text_is_reward = check_task_group_verifiable(filtered_rl_ds, domain, task_group, task_type)
    print(f"{domain}/{task_group}/{task_type}: {answer_text_is_reward}")
    print(len(filtered_rl_ds))
    
    for row in filtered_rl_ds.select(range(4)):
        print(row)
    print("_______________________\n\n")

biology_qa bio_mcq None


Filter: 100%|██████████| 71011/71011 [00:00<00:00, 147534.84 examples/s]


9998
biology_qa/bio_mcq/None: True
9998
{'prompt': [{'content': "Answer the following multiple choice question. The last line of your response should be of the following format: 'ANSWER: LETTER' (without quotes) where LETTER is one of A,B,C,D,E,F,G,H,I,J. For example, if A is correct, write exactly 'ANSWER: A'. Do not include a dollar sign or other formatting around the letter. Think step by step before answering.\n\nQuestion:\nA 25-year-old male presents with multiple colorectal adenomas and glioblastoma multiforme. Genetic testing reveals a germline mutation in an autosomal dominant gene. Which combination of findings is most characteristic of Turcot Syndrome type 1 (Lynch syndrome-associated) rather than type 2 (FAP-associated)?\nOptions:\nA) Medulloblastoma with CHRPE lesions and desmoid tumors\nB) Pilocytic astrocytoma with hyperplastic polyposis\nC) Pineoblastoma with Peutz-Jeghers type polyps\nD) Meningioma with attenuated polyposis and APC mutation\nE) Ependymoma with juvenile-

In [ ]:
# checking the verifiability of answers inside the instruction_folowing domain
domain_index = 3
domain = wanted_domains[domain_index]
verifiable_task_groups = []
task_types_counter = task_group_types_counters[domain_index]

for (task_group, task_type), _ in task_types_counter:
    print(domain, task_group, task_type)
    if task_group in verifiable_task_groups:
                continue
                
    filtered_rl_ds = rl_ds.filter(lambda row: row["domain"]==domain and row["task_group"]==task_group and row["task_type"]==task_type)
    print(len(filtered_rl_ds))
    answer_text_is_reward = check_task_group_verifiable(filtered_rl_ds, domain, task_group, task_type)
    print(f"{domain}/{task_group}/{task_type}: {answer_text_is_reward}")
    print(len(filtered_rl_ds))
    
    for row in filtered_rl_ds.select(range(4)):
        print(row)
    print("_______________________\n\n")

instruction_following if None


Filter: 100%|██████████| 71011/71011 [00:00<00:00, 137761.11 examples/s]


4000
instruction_following/if/None: False
4000
{'prompt': [{'content': "Please follow these instructions in your response:\n- In your response, the letter b should appear at least 3 times.\n- There should be 4 paragraphs. Paragraphs and only paragraphs are separated with each other by two new lines as if it was '\\n\\n' in python. Paragraph 2 must start with word doubt.\n\n帮我写一份锐捷AP设备产品介绍以及对比TP设备，企业为什么更适合选择锐捷设备的优势的分析宣传", 'role': 'user'}], 'answer_text': None, 'task_group': 'if', 'task_type': None, 'verifier_metadata': {'prompt': "Please follow these instructions in your response:\n- In your response, the letter b should appear at least 3 times.\n- There should be 4 paragraphs. Paragraphs and only paragraphs are separated with each other by two new lines as if it was '\\n\\n' in python. Paragraph 2 must start with word doubt.\n\n帮我写一份锐捷AP设备产品介绍以及对比TP设备，企业为什么更适合选择锐捷设备的优势的分析宣传", 'instruction_id_list': ['keywords:letter_frequency', 'length_constraints:nth_paragraph_first_word'], 'kwargs': 

In [45]:
# function that checks for missing rows in required fields
def audit_rows(ds, required_fields):
    missing = Counter()
    row_indices = []
    for i, item in enumerate(ds):
        row = cast(dict[str, Any], item)
        for field in required_fields:
            value = row.get(field)
            if value==None or value=="" or value==[]:
                missing[field] += 1
                row_indices.append(i)
    print(f"Missing Count: {missing}")
    return missing, row_indices

In [46]:
#check for missing rows in all our datasets
# dna_computation ds
item =  next(iter(computation_ds))
comp_row = cast(dict[str, Any], item)
comp_fields = comp_row.keys()
audit_rows(computation_ds, comp_fields)

# sequence_classification ds
item =  next(iter(classification_ds))
classif_row = cast(dict[str, Any], item)
classif_fields = classif_row.keys()
audit_rows(classification_ds, classif_fields)

# rl ds
rl_fields = ["domain", "task_group", "prompt"]
audit_rows(rl_ds, rl_fields)

Missing Count: Counter()
Missing Count: Counter()
Missing Count: Counter()


(Counter(), [])